# extract raw data

## import libraries

In [4]:
import requests
import pandas as pd
from datetime import date, timedelta
from tqdm import tqdm

## Load MBTA LAMP Data

Fetches daily subway on-time performance data from the
[MBTA LAMP open data portal](https://performancedata.mbta.com) for a given date range.

Each day is downloaded as a `.parquet` file, filtered to remove rows with missing
`travel_time_seconds` or `dwell_time_seconds`, and concatenated into a single DataFrame.
Dates where data is unavailable are skipped gracefully.

In [5]:
def get_lamp_data(start_date, end_date):
    dfs = []
    current = start_date
    
    # Iterate day by day from start to end date
    while current <= end_date:
        date_str = current.strftime("%Y-%m-%d")

        # Construct MBTA LAMP parquet URL for the current date
        url = f"https://performancedata.mbta.com/lamp/subway-on-time-performance-v1/{date_str}-subway-on-time-performance-v1.parquet"
        
        try:
            df = pd.read_parquet(url)
            df["service_date"] = date_str

            # Drop rows with missing travel or dwell times
            df = df[
                ~df['travel_time_seconds'].isna() &
                ~df['dwell_time_seconds'].isna()]

            dfs.append(df)
            print(f"✓ Loaded {date_str} — {len(df)} rows")

        except Exception as e:
            # Skip dates where data is unavailable or download fails
            print(f"✗ Skipping {date_str}: {e}")
        
        current += timedelta(days=1)
    
    # Combine all daily dataframes, or return empty if none loaded
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

In [6]:
# data from oct 2025
df_oct25 = get_lamp_data(
    start_date = date(2025, 10, 1),
    end_date   = date(2025, 10, 31)
)

print(df_oct25.shape)

# data from oct 2024
df_oct24 = get_lamp_data(
    start_date = date(2024, 10, 1),
    end_date   = date(2024, 10, 31)
)

print(df_oct24.shape)

df = pd.concat([df_oct25, df_oct24], ignore_index=True).reset_index(drop=True)

✓ Loaded 2025-10-01 — 41643 rows
✓ Loaded 2025-10-02 — 40046 rows
✓ Loaded 2025-10-03 — 40704 rows
✓ Loaded 2025-10-04 — 30517 rows
✓ Loaded 2025-10-05 — 28400 rows
✓ Loaded 2025-10-06 — 41411 rows
✓ Loaded 2025-10-07 — 41771 rows
✓ Loaded 2025-10-08 — 42511 rows
✓ Loaded 2025-10-09 — 41083 rows
✓ Loaded 2025-10-10 — 42220 rows
✓ Loaded 2025-10-11 — 28800 rows
✓ Loaded 2025-10-12 — 27841 rows
✓ Loaded 2025-10-13 — 28719 rows
✓ Loaded 2025-10-14 — 42070 rows
✓ Loaded 2025-10-15 — 41222 rows
✓ Loaded 2025-10-16 — 41304 rows
✓ Loaded 2025-10-17 — 43297 rows
✓ Loaded 2025-10-18 — 33123 rows
✓ Loaded 2025-10-19 — 31463 rows
✓ Loaded 2025-10-20 — 41356 rows
✓ Loaded 2025-10-21 — 42533 rows
✓ Loaded 2025-10-22 — 41327 rows
✓ Loaded 2025-10-23 — 41444 rows
✓ Loaded 2025-10-24 — 40777 rows
✓ Loaded 2025-10-25 — 29724 rows
✓ Loaded 2025-10-26 — 27701 rows
✓ Loaded 2025-10-27 — 37613 rows
✓ Loaded 2025-10-28 — 38129 rows
✓ Loaded 2025-10-29 — 35322 rows
✓ Loaded 2025-10-30 — 37443 rows
✓ Loaded 2

## breakdown of every column:

**Stop Info**
| Column | Description |
|--------|-------------|
| `stop_sequence` | Order of this stop within the trip (1 = first stop) |
| `stop_id` | Unique ID of the specific stop/platform |
| `parent_station` | Station the stop belongs to (e.g. Park St has multiple platforms) |
| `stop_count` | Total number of stops in the trip |

**Timestamps**
| Column | Description |
|--------|-------------|
| `stop_timestamp` | Actual time train **arrived** at stop (Unix seconds) |
| `move_timestamp` | Actual time train **departed** stop (Unix seconds) |
| `start_time` | Actual time the trip started |
| `service_date` | Operating date (note: trips after midnight still belong to previous service date) |

**Travel & Performance**
| Column | Description |
|--------|-------------|
| `travel_time_seconds` | Actual time taken to travel from previous stop to this stop |
| `dwell_time_seconds` | Actual time spent at this stop (move - stop timestamp) |
| `headway_trunk_seconds` | Actual time since last train on the **main line** |
| `headway_branch_seconds` | Actual time since last train on the **branch** |

**Scheduled (baseline to compare against)**
| Column | Description |
|--------|-------------|
| `scheduled_arrival_time` | Scheduled arrival at this stop (Unix seconds) |
| `scheduled_departure_time` | Scheduled departure from this stop (Unix seconds) |
| `scheduled_travel_time` | Scheduled travel time from previous stop |
| `scheduled_headway_trunk` | Scheduled headway on main line |
| `scheduled_headway_branch` | Scheduled headway on branch |

**Trip & Route Info**
| Column | Description |
|--------|-------------|
| `trip_id` | Unique ID for this specific trip/run |
| `route_id` | Route (e.g. `Red`, `Orange`, `Green-B`) |
| `branch_route_id` | Branch of the route (e.g. `Green-B`, `Green-C`) |
| `trunk_route_id` | Main trunk line shared by branches (e.g. `Green`) |
| `direction_id` | `0` = outbound, `1` = inbound |
| `direction` | Text direction e.g. `"Southbound"` |
| `direction_destination` | Final destination e.g. `"Alewife"` |

**Vehicle Info**
| Column | Description |
|--------|-------------|
| `vehicle_id` | Unique ID of the physical train/vehicle |
| `vehicle_label` | Human-readable train number |
| `vehicle_consist` | Which cars make up the train |

In [7]:
# Timestamp columns stored as Unix seconds — convert to Eastern Time for readability
timestamp_cols = [
    "stop_timestamp",
    "move_timestamp", 
    "scheduled_arrival_time",
    "scheduled_departure_time",
    "start_time"
]

# Convert from Unix seconds (UTC) to timezone-aware Eastern Time
for col in timestamp_cols:
    df[col] = pd.to_datetime(df[col], unit="s", utc=True).dt.tz_convert("America/New_York")

## Map Stop IDs to Stop Names

Downloads the MBTA static stops reference table and builds a `stop_id → stop_name`
lookup dictionary, used to resolve human-readable station names from raw stop IDs.

In [8]:
stops = pd.read_parquet("https://performancedata.mbta.com/lamp/tableau/rail/LAMP_static_stops.parquet")
stop_names = stops.set_index('stop_id')['stop_name'].to_dict()

In [9]:
stops.to_csv(r'./data/stop_name.csv', index=False)

## save data frame

In [10]:
# set stop name
df['stop_id'] = df['stop_id'].astype(str)
df['stop_name'] = df['stop_id'].apply(lambda x: stop_names[x])

In [11]:
df.to_csv(r'.\data\processed_oct_Data.csv',
          index=False)